In [ ]:
import pandas as pd
import numpy as np

# ============================================================================
# 1. LOAD CLEANED DATA & PRE-PROCESS NEW TABLES
# ============================================================================
base_path = '..\\ACL_9th-F1\\F1_clean\\'

results = pd.read_csv('..\\ACL_9th-F1\\F1_clean\\results_clean_with_rank.csv')
races = pd.read_csv(base_path + 'races_clean.csv')
driver_standings = pd.read_csv(base_path + 'driver_standings_clean.csv')
qualifying = pd.read_csv('..\\ACL_9th-F1\\F1\\' + 'qualifying.csv')
drivers = pd.read_csv(base_path + 'drivers_clean.csv')
circuits = pd.read_csv(base_path + 'circuits_clean.csv')
constructors = pd.read_csv(base_path + 'constructors_clean.csv')
results['podium'] = (results['positionOrder'] <= 3).astype(int)
# Load the new tables
lap_times = pd.read_csv('..\\ACL_9th-F1\\F1\\lap_times.csv')
pit_stops = pd.read_csv('..\\ACL_9th-F1\\F1\\pit_stops.csv')
constructor_standings = pd.read_csv('..\\ACL_9th-F1\\F1\\constructor_standings.csv')

# Pre-process 'rank' (fastest lap rank) to numeric, converting '\N' to NaN
if 'rank' in results.columns:
    results['rank'] = pd.to_numeric(results['rank'].replace(r'\N', np.nan), errors='coerce')

# Aggregate average lap position per driver per race
race_lap_avg = lap_times.groupby(['raceId', 'driverId'])['position'].mean().reset_index(name='race_avg_lap_pos')

# Aggregate average pit stop duration per driver per race
pit_avg = pit_stops.groupby(['raceId', 'driverId'])['milliseconds'].mean().reset_index(name='race_avg_pit_ms')

# =============================================================================
# 2. DRIVER FORM FEATURES (Carries over between seasons)
# =============================================================================
# Merge results with races and new aggregates to sort chronologically
driver_form = results[['raceId', 'driverId', 'positionOrder', 'podium', 'rank']].copy()
driver_form = driver_form.merge(races[['raceId', 'race_year', 'race_round']], on='raceId')
driver_form = driver_form.merge(race_lap_avg, on=['raceId', 'driverId'], how='left')
driver_form = driver_form.merge(pit_avg, on=['raceId', 'driverId'], how='left')

# Sort chronologically by driver
driver_form = driver_form.sort_values(by=['driverId', 'race_year', 'race_round'])

# --- RECENT FORM (Shifted to exclude current race) ---
driver_form['driver_prev_position'] = driver_form.groupby('driverId')['positionOrder'].shift(1)
driver_form['driver_prev_podium'] = driver_form.groupby('driverId')['podium'].shift(1).fillna(0)

g = driver_form.groupby('driverId')   # every calculation below stays inside one driver's own races

# Rolling 5-Race Averages (shift(1) = only earlier races)
driver_form['driver_avg_pos_last_5'] = g['positionOrder'].transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
driver_form['driver_avg_lap_pos_last_5'] = g['race_avg_lap_pos'].transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
driver_form['driver_avg_pit_ms_last_5'] = g['race_avg_pit_ms'].transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
driver_form['driver_avg_speed_rank_last_5'] = g['rank'].transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())

# --- CAREER TOTALS (all earlier races of this driver) ---
driver_form['driver_career_avg_pos'] = g['positionOrder'].transform(lambda s: s.shift(1).expanding().mean())
driver_form['driver_career_podiums'] = g['podium'].transform(lambda s: s.shift(1).cumsum()).fillna(0)

# Isolate columns for final merge
driver_features_to_keep = [
    'raceId', 'driverId', 'driver_prev_position', 'driver_prev_podium', 
    'driver_avg_pos_last_5', 'driver_avg_lap_pos_last_5', 'driver_avg_pit_ms_last_5', 
    'driver_avg_speed_rank_last_5', 'driver_career_avg_pos', 'driver_career_podiums'
]
driver_form = driver_form[driver_features_to_keep]

# =============================================================================
# 3. DRIVER STANDINGS FEATURES (Resets to 0 every season)
# =============================================================================
ds = driver_standings.merge(races[['raceId', 'race_year', 'race_round']], on='raceId')
ds = ds.sort_values(by=['driverId', 'race_year', 'race_round'])

ds['pre_race_points'] = ds.groupby(['driverId', 'race_year'])['driver_points_cumm'].shift(1).fillna(0)
ds['pre_race_wins'] = ds.groupby(['driverId', 'race_year'])['driver_wins_cumm'].shift(1).fillna(0)
ds['won_previous_race'] = ds.groupby(['driverId', 'race_year'])['pre_race_wins'].diff().fillna(0)

ds = ds[['raceId', 'driverId', 'pre_race_points', 'pre_race_wins', 'won_previous_race']]

# =============================================================================
# 3.5 CONSTRUCTOR FORM & STANDINGS (NEW)
# =============================================================================
# --- CONSTRUCTOR FORM ---
constructor_form = results[['raceId', 'constructorId', 'positionOrder', 'podium']].copy()
constructor_form = constructor_form.merge(races[['raceId', 'race_year', 'race_round']], on='raceId')

# Best finish and total podiums for the team per race
constructor_form = constructor_form.groupby(['raceId', 'constructorId', 'race_year', 'race_round']).agg({
    'positionOrder': 'min',
    'podium': 'sum'
}).reset_index()

constructor_form = constructor_form.sort_values(by=['constructorId', 'race_year', 'race_round'])
gc = constructor_form.groupby('constructorId')   # every calculation stays inside one team's own races

constructor_form['constructor_prev_pos'] = gc['positionOrder'].shift(1)
constructor_form['constructor_avg_pos_last_5'] = gc['positionOrder'].transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
constructor_form['constructor_career_podiums'] = gc['podium'].transform(lambda s: s.shift(1).cumsum()).fillna(0)

constructor_form = constructor_form[['raceId', 'constructorId', 'constructor_prev_pos', 'constructor_avg_pos_last_5', 'constructor_career_podiums']]

# --- CONSTRUCTOR STANDINGS ---
cs = constructor_standings.merge(races[['raceId', 'race_year', 'race_round']], on='raceId')
cs = cs.sort_values(by=['constructorId', 'race_year', 'race_round'])

# Adjust column names 'constructor_points_cumm'/'constructor_wins_cumm' if they differ in your CSV
cs['pre_race_constructor_points'] = cs.groupby(['constructorId', 'race_year'])['points'].shift(1).fillna(0)
cs['pre_race_constructor_wins'] = cs.groupby(['constructorId', 'race_year'])['wins'].shift(1).fillna(0)

cs = cs[['raceId', 'constructorId', 'pre_race_constructor_points', 'pre_race_constructor_wins']]

# =============================================================================
# 4. BUILD THE BASE TRAINING TABLE (Strict Leakage Control)
# =============================================================================
final_df = results[['raceId', 'driverId', 'constructorId', 'grid', 'podium']].copy()

final_df = final_df.merge(races[['raceId', 'race_year', 'race_round', 'circuitId']], on='raceId', how='left')

final_df = final_df.merge(qualifying[['raceId', 'driverId', 'position']], on=['raceId', 'driverId'], how='left')
final_df.rename(columns={'position': 'quali_position'}, inplace=True)

final_df = final_df.merge(drivers[['driverId', 'drivers_nationality']], on='driverId', how='left')
final_df.rename(columns={'drivers_nationality': 'driver_nationality'}, inplace=True)

final_df = final_df.merge(circuits[['circuitId', 'country']], on='circuitId', how='left')
final_df.rename(columns={'country': 'circuit_country'}, inplace=True)

final_df = final_df.merge(constructors[['constructorId', 'name']], on='constructorId', how='left')
final_df.rename(columns={'name': 'constructor_name'}, inplace=True)

final_df = final_df.merge(driver_form, on=['raceId', 'driverId'], how='left')
final_df = final_df.merge(ds, on=['raceId', 'driverId'], how='left')

# Add Constructor details
final_df = final_df.merge(constructor_form, on=['raceId', 'constructorId'], how='left')
final_df = final_df.merge(cs, on=['raceId', 'constructorId'], how='left')

# =============================================================================
# 5. HOME RACE FEATURE ENGINEERING
# =============================================================================
# Dictionary mapping F1 nationalities to circuit countries
nationality_to_country = {
    'British': 'UK', 'German': 'Germany', 'Spanish': 'Spain', 'Finnish': 'Finland',
    'Japanese': 'Japan', 'French': 'France', 'Polish': 'Poland', 'Brazilian': 'Brazil',
    'Italian': 'Italy', 'Australian': 'Australia', 'Austrian': 'Austria', 'American': 'USA',
    'Dutch': 'Netherlands', 'Colombian': 'Colombia', 'Portuguese': 'Portugal', 
    'Canadian': 'Canada', 'Indian': 'India', 'Hungarian': 'Hungary', 'Irish': 'Ireland',
    'Danish': 'Denmark', 'Argentine': 'Argentina', 'Belgian': 'Belgium', 
    'South African': 'South Africa', 'Swiss': 'Switzerland', 'Swedish': 'Sweden',
    'Mexican': 'Mexico', 'Russian': 'Russia', 'Chinese': 'China', 
    'Monegasque': 'Monaco', 'New Zealander': 'New Zealand'
}

final_df['driver_home_country'] = final_df['driver_nationality'].map(nationality_to_country).fillna(final_df['driver_nationality'])
# final_df['circuit_country'] = final_df['circuit_country'].replace({'United States': 'USA'})
final_df['is_home_race'] = (final_df['driver_home_country'] == final_df['circuit_country']).astype(int)
final_df = final_df.drop(columns=['driver_home_country'])

# =============================================================================
# 6. FINAL CLEANUP & EXPORT
# =============================================================================
# 6A. Grid position with grid 0 (pit-lane start / no grid slot) moved behind the last car of THAT race,
#     then use it to fill missing qualifying positions
last_slot = final_df.groupby('raceId')['grid'].transform('max')
final_df['grid_pos'] = final_df['grid'].where(final_df['grid'] != 0, last_slot + 1)
final_df['quali_position'] = final_df['quali_position'].fillna(final_df['grid_pos'])

# 6B. Fill zero-baselines for debuts
final_df['pre_race_points'] = final_df['pre_race_points'].fillna(0)
final_df['pre_race_wins'] = final_df['pre_race_wins'].fillna(0)
final_df['won_previous_race'] = final_df['won_previous_race'].fillna(0)
final_df['driver_prev_podium'] = final_df['driver_prev_podium'].fillna(0)
final_df['pre_race_constructor_points'] = final_df['pre_race_constructor_points'].fillna(0)
final_df['pre_race_constructor_wins'] = final_df['pre_race_constructor_wins'].fillna(0)

# 6C. Fill rookie/debut Form with "back of pack" baseline (20)
final_df['driver_prev_position'] = final_df['driver_prev_position'].fillna(20)
final_df['driver_avg_pos_last_5'] = final_df['driver_avg_pos_last_5'].fillna(20)
final_df['driver_career_avg_pos'] = final_df['driver_career_avg_pos'].fillna(20)
final_df['constructor_prev_pos'] = final_df['constructor_prev_pos'].fillna(20)
final_df['constructor_avg_pos_last_5'] = final_df['constructor_avg_pos_last_5'].fillna(20)

# 6D. Fill missing lap/pit stop data (due to decades of missing coverage)
final_df['driver_avg_speed_rank_last_5'] = final_df['driver_avg_speed_rank_last_5'].fillna(20)
final_df['driver_avg_lap_pos_last_5'] = final_df['driver_avg_lap_pos_last_5'].fillna(20)
# Fill missing pit stop milliseconds with a heavy penalty (e.g. 40000ms / 40s)
final_df['driver_avg_pit_ms_last_5'] = final_df['driver_avg_pit_ms_last_5'].fillna(40000)

# 6E. Championship points relative to the leader BEFORE this race (same 0-1 scale in every season)
leader = final_df.groupby('raceId')['pre_race_points'].transform('max')
final_df['points_share'] = np.where(leader > 0, final_df['pre_race_points'] / leader, 0)    # round 1: nobody has points -> 0

leader_c = final_df.groupby('raceId')['pre_race_constructor_points'].transform('max')
final_df['constructor_points_share'] = np.where(leader_c > 0, final_df['pre_race_constructor_points'] / leader_c, 0)

final_df.to_csv(base_path + 'FINAL_training_table.csv', index=False)
print(f"Final Training Table built successfully! Shape: {final_df.shape}")
print("Columns:", final_df.columns.tolist())


### We are going to discard the years before 2004 due to the absence of qualifying systems in these years

In [ ]:
final_df = final_df[final_df['race_year'] > 2003]

In [ ]:
final_df.isnull().sum()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Define your two X features and your single Y target variable
selected_columns = ["quali_position", "grid", "podium", "driver_avg_pos_last_5", "pre_race_points", "pre_race_wins",
                    "won_previous_race", "is_home_race", "driver_prev_position", "driver_prev_podium",
                    "driver_avg_lap_pos_last_5", "driver_avg_pit_ms_last_5", "driver_avg_speed_rank_last_5",
                    "driver_career_avg_pos", "driver_career_podiums", "constructor_prev_pos",
                    "constructor_avg_pos_last_5", "constructor_career_podiums", "pre_race_constructor_points",
                    "pre_race_constructor_wins", "points_share", "constructor_points_share"]

# 2. Calculate the correlation matrix
matrix = final_df[selected_columns].corr()

print("--- Correlation Matrix (X features vs Y target) ---")
print(matrix)

# 3. Plot a clean heatmap to visualize the relationships
plt.figure(figsize=(20, 15))
sns.heatmap(matrix, annot=True, cmap="coolwarm", vmin=-1, vmax=1, fmt=".4f")
plt.title("Correlation Matrix: X Features vs Y (Podium)")
plt.tight_layout()
plt.show()


In [ ]:
final_df['race_year'].value_counts().sort_index().plot(kind='bar', figsize=(16, 4))
plt.title('Number of races per season')
plt.xlabel('season'); plt.ylabel('races')
plt.show()
print('first season:', final_df['race_year'].min(), '| last season:', final_df['race_year'].max(),
      '| seasons:', final_df['race_year'].nunique())

## Feature Selection

### 1. Reason for the chosen features (with evidence)
Every candidate must be **known before the race starts** (after qualifying). The evidence is the correlation with
`podium` on the modelling seasons (2004+, 8,650 rows, from the heatmap above) and the podium-rate plots below.

| Group | Feature | r with podium | Why it is relevant | Decision |
|---|---|---|---|---|
| Starting position | `quali_position` | −0.50 | Pace on the same weekend | Keep |
| | `grid` | −0.48 | Actual starting slot (after penalties) | Keep |
| Driver recent form | `driver_avg_lap_pos_last_5` | −0.48 | Average running position in the last 5 races | Keep |
| | `driver_avg_pos_last_5` | −0.47 | Average finish in the last 5 races | Keep |
| | `driver_avg_speed_rank_last_5` | −0.45 | Fastest-lap rank in the last 5 races (raw pace) | Keep |
| | `driver_prev_podium` | +0.41 | Podium in the previous race | Keep |
| | `driver_prev_position` | −0.35 | Finish in the previous race | Keep |
| | `won_previous_race` | +0.30 | Won the previous race | Keep |
| Driver career | `driver_career_avg_pos` | −0.38 | Long-term level of the driver | Keep |
| | `driver_career_podiums` | +0.29 | Proven podium history | Keep |
| Driver standings | `pre_race_points` | +0.43 | Season performance so far | Keep |
| | `pre_race_wins` | +0.43 | Season wins so far | Keep |
| Team form | `constructor_avg_pos_last_5` | −0.46 | Car strength over the last 5 races | Keep |
| | `constructor_prev_pos` | −0.40 | Team's best finish in the previous race | Keep |
| | `constructor_career_podiums` | +0.24 | Team's long-term history | Keep |
| Team standings | `pre_race_constructor_wins` | +0.45 | Team wins this season | Keep |
| | `pre_race_constructor_points` | +0.41 | Team points this season | Keep |
| Weak | `driver_avg_pit_ms_last_5` | +0.01 | Average pit-stop time in the last 5 races | **Drop** |
| | `is_home_race` | +0.00 | Racing in the driver's home country | **Drop** (kept for Q2 only) |

**Why the two features are dropped**
- `driver_avg_pit_ms_last_5`: pit-stop data only exists from 2011, so **all 2004–2010 rows (32% of the table)**
  hold the fill value 40,000 ms instead of a real time. On real values the relationship is weak and not one-directional
  (podium rate 21% → 10% → 16% from shortest to longest stops), and the linear correlation is +0.01.
- `is_home_race`: only 3.9% of rows are home races, and their podium rate is the same as everyone else's
  (14.7% vs 14.3%), which matches our answer to Q2.

**Redundancy (correlation between features ≥ 0.8)**: `pre_race_points` ~ `pre_race_constructor_points` (0.96),
`grid` ~ `quali_position` (0.93), `driver_avg_pos_last_5` ~ `driver_avg_lap_pos_last_5` (0.91). They are kept because
they still differ in meaningful cases (e.g. grid penalties: grid ≠ qualifying for a share of drivers; a driver can be
much better or worse than his team-mate). Tree models handle this overlap; we do not interpret logistic-regression
coefficients one by one.

### 2. Plotting the effect of the features
For each feature, the rows are split into 10 equal-size groups (deciles) from the lowest to the highest value, and
we plot the share of podiums in each group. A useful feature shows bars clearly above and below the red line
(overall podium rate, 14.3%).

In [ ]:
import matplotlib.pyplot as plt

FEATURES = ["quali_position", "grid", "driver_avg_pos_last_5", "driver_avg_lap_pos_last_5",
            "driver_avg_speed_rank_last_5", "driver_prev_position", "driver_prev_podium",
            "driver_career_avg_pos", "driver_career_podiums", "pre_race_points", "pre_race_wins",
            "won_previous_race", "constructor_prev_pos", "constructor_avg_pos_last_5",
            "constructor_career_podiums", "pre_race_constructor_points", "pre_race_constructor_wins",
            "driver_avg_pit_ms_last_5", "is_home_race"]

base_rate = final_df['podium'].mean()
fig, axes = plt.subplots(4, 5, figsize=(26, 17))
for ax, f in zip(axes.ravel(), FEATURES):
    d = final_df[[f, 'podium']]
    if d[f].nunique() <= 10:                       # few values: one bar per value
        rate = d.groupby(f)['podium'].mean()
        xlabel = 'value'
    else:                                          # many values: 10 equal-size groups (deciles), low -> high
        groups = pd.qcut(d[f], 10, duplicates='drop')
        rate = d.groupby(groups, observed=True)['podium'].mean()
        rate.index = range(1, len(rate) + 1)
        xlabel = 'decile (1 = lowest values)'
    rate.plot(kind='bar', ax=ax, color='steelblue')
    ax.axhline(base_rate, color='red', linestyle='--', linewidth=1)   # overall podium rate
    ax.set_title(f'{f}\nr = {final_df[f].corr(final_df["podium"]):+.2f}', fontsize=11)
    ax.set_xlabel(xlabel); ax.set_ylabel('podium rate'); ax.tick_params(axis='x', labelsize=7, rotation=0)
for ax in axes.ravel()[len(FEATURES):]:
    ax.axis('off')
plt.suptitle('Podium rate across the range of each feature (red line = overall podium rate)', fontsize=16)
plt.tight_layout(); plt.show()

**Observations**
- **Starting position is decisive:** the best decile of `grid` / `quali_position` reaches a ~65% podium rate, and from
  the 4th decile on it is almost 0.
- **Recent form behaves the same way:** the best decile of `driver_avg_pos_last_5`, `driver_avg_lap_pos_last_5`,
  `driver_avg_speed_rank_last_5` and `constructor_avg_pos_last_5` is at 55–60%, the worst near 0.
- **Yes/no features split clearly:** a podium in the previous race → ~50% podium rate (vs ~8%); winning the previous
  race → ~63% (vs ~12%).
- **Standings rise steadily:** the highest groups of `pre_race_points` / `pre_race_constructor_points` reach ~50%.
- **`driver_prev_position` rises slightly again at the worst values:** a retirement in the previous race gives a high
  position number even to fast drivers, so one bad race is a noisy signal — the 5-race average is cleaner.
- **`constructor_career_podiums` is not one-directional** (the 6th group is far above the 7th): a long history counts
  old successful teams that are no longer fast, so the recent team-form features carry the stronger signal.
- **`driver_avg_pit_ms_last_5` and `is_home_race` stay around the red line** → dropped (see the table above).

### 3. Leakage-safe form features (only past races)
The prediction is made after qualifying and before the race, so **no information from the race itself** may enter a
feature.

- **Excluded from features:** everything produced by the race being predicted (`positionOrder`, points, laps, status,
  that race's fastest-lap rank, lap times and pit stops).
- **Form and career features:** within each driver (or team), sorted by `race_year` and `race_round`, every value is
  computed from `shift(1)`, i.e. only from earlier races:
  `groupby('driverId')[col].transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())`.
  The shift, the rolling window and the running totals all stay **inside the group**. An earlier version applied
  `.rolling()` / `.expanding()` after the grouping had ended, so the windows ran across drivers. This made
  `driver_career_podiums` correlate *negatively* with podium (−0.09); after the fix it is +0.29.
- **Standings features:** the cumulative points and wins are shifted by one race within the same driver/team and
  season, so they are the standings **before** the race (0 before the first race of a season).
- **Same-weekend information:** only `grid` and `quali_position`, both fixed before the start.
- **The season filter (2004+) is applied after the features are built**, so earlier seasons still count as history.

**Check:** for 200 random rows, the last-5 average finish and the career podium count are recomputed by hand from
strictly earlier races only, and compared with the table.

In [ ]:
# Leakage check: recompute two form features by hand for random rows, using ONLY races that happened earlier
hist = results[['raceId', 'driverId', 'positionOrder', 'podium']].merge(races[['raceId', 'race_year', 'race_round']], on='raceId')
hist['order'] = hist['race_year'] * 100 + hist['race_round']                  # time order: season, then round

sample = final_df.sample(200, random_state=42)
checks = []
for _, row in sample.iterrows():
    now = row['race_year'] * 100 + row['race_round']
    past = hist[(hist['driverId'] == row['driverId']) & (hist['order'] < now)].sort_values('order')   # strictly earlier races
    exp_avg5 = past['positionOrder'].tail(5).mean() if len(past) else 20       # 20 = notebook's fill for debuts
    exp_career = past['podium'].sum()
    checks.append({'avg_pos_last_5 matches': np.isclose(row['driver_avg_pos_last_5'], exp_avg5),
                   'career_podiums matches': row['driver_career_podiums'] == exp_career})
checks = pd.DataFrame(checks)
print('rows checked:', len(checks))
print(checks.mean().rename('share matching'))

**Result:** 200 / 200 rows match for both features (share = 1.0). Every value is computed only from races before the
one being predicted, so the form features are leakage-safe.